# Datamine EXPMMW Process Example

This notebook demonstrates how to configure and run the **`expmmw`** process wrapper in `dmstudio`.

## Process Description

## EXPMMW

This process expands a set of planar perimeters according to a user defined minimum mining width (MMW). This will allow perimeters which describe geological or grade boundaries to be expanded into a mineable entity.

The perimeters must be planar, but not necessarily in one of the orthogonal planes.

The process first calculates the plane of each perimeter, and transforms the coordinates into that plane; the perimeter is then tested against the MMW constraint and adjusted as necessary; finally the expanded perimeter is transformed back into its original plane. The test against MMW is done by first calculating the centre line of the perimeter.

A minimum perimeter is created based on this centre line with the MMW being measured perpendicular to the line. Each side of the minimum perimeter is then 0.5 x MMW from the centre line. The original perimeter is then tested against the minimum perimeter and is adjusted, if necessary, to ensure that no part of it lies within the minimum perimeter.

The expanded perimeter is then written to the output file. All perimeters will automatically be checked for crossovers and consecutive duplicate points Duplicate points will be removed. Malformed perimeters will be reported, but they will not be processed or included in the output file. Input perimeters may be either open or closed. Output perimeters will be closed. It is not possible to define an algorithm which will deal with every shape and size of perimeter. Therefore it has been implemented as a two pass process, controlled by parameter **MODE**.

User interaction is required between the passes to check and if necessary edit the strings. Pass 1 (**MODE** =1) takes as input the original perimeters from the **PERIMIN** file and creates the **PERIMOUT** file containing the original perimeters, the centre line strings and the expanded perimeters. The strings in this file can then be checked interactively in the graphics window.

If **EXPMMW** has not produced a satisfactory expanded perimeter then the centre line string should be edited appropriately. Two extra fields will be created in the **PERIMOUT** file:

**PTYPE** \- the perimeter type field. This expects values:

* 0 \- the original perimeter

* 1 \- centre line string

* 2 \- expanded perimeter

**PORIG** \- this field contains the **PVALUE** of the original perimeter.

If the **PERIMIN** file contains a field then these values will be carried through to the **PERIMOUT** file. The value of will be incremented by 1 for centre lines and by 2 for expanded perimeters. If the **PERIMIN** file does not contain a field, then a field will be created in the output file. The of each original perimeter will be assigned a value of 1, 2 for centre lines and 3 for the expanded perimeters. Editing the centre line strings in the graphics window will usually only involve small changes to the centre line strings near the intersection with the original perimeters.

Pass 2 through process **EXPMMW** takes as input a file containing both the original perimeters and the centre lines. Fields **PTYPE** and **PORIG** are required in the **PERIMIN** file. In practice it is probably best for the PERIMIN file to contain all perimeters even if the centre line did not require editing. In this way all the expanded perimeters are kept in the same file.

### Input Files:

* **perimin** (String):
  The input perimeter file. If **MODE** =1 then the fields required are **XP** ,**YP**
  ,**ZP** , **PTN** and **PVALUE** ie standard perimeter format. Any other fields in this
  file will not be copied to the output file. All valid perimeters in the file will be
  used. If **MODE** =2 or 3 then it must also contain **PTYPE** and **PORIG** fields. In
  addition the file must be sorted on the keyfields **PORIG** , **PTYPE** and **PTN**.
  Required=Yes

### Output Files:

* **perimout** (String):
  The output perimeter file containing the expanded perimeters. It may also contain a
  copy of the original perimeters and the centre line strings depending on the value of
  parameter **MODE**. The file will contain the standard perimter fields **XP** ,**YP**
  ,**ZP** ,**PTN** and **PVALUE** plus ,**PTYPE** and **PORIG**. Malformed input
  perimeters will be reported but not processed. In place processing is not permitted ie
  **PERIMIN** and **PERIMOUT** must be different files.
  Required=Yes

### Fields:

### Parameters:

* **mmw**:
  The minimum mining width, measured perpendicular to the centre line of the perimeter.
  Any point on the expanded perimeter will be at least 0.5 x **MMW** from the centre line.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **pinc**:
  Numeric increment for **PVALUE** s for centre line string and adjusted perimeter as
  written to the **PERIMOUT** file. If the **PVALUE** of the input perimeter is P then its
  centre line string will have a **PVALUE** of P+PINC and the expanded perimeter will have
  a **PVALUE** of P+2xPINC. The default value is (0.1).
  Range=Undefined
  Values=Undefined
  Default=0.1
  Required=No

* **mode**:
  This parameter defines the contents of both the **PERIMIN** and **PERIMOUT** files. It
  has values: Options: 1: **PERIMIN** : original perimeters **PERIMOUT** : original
  perimeters centre lines expanded perimeters.; 2: **PERIMIN** : original perimeters
  centre lines **PERIMOUT** : original perimeters centre lines expanded perimeters.; 3:
  **PERIMIN** : original perimeters centre lines **PERIMOUT** : expanded perimeters For
  pass 1 MODE must be set to 1. For pass 2 **MODE** must be either 2 or 3. The default
  value of **MODE** is (1).
  Range=1,3
  Values=1,2,3
  Default=1
  Required=No

* **nodiag**:
  This parameter specifies whether the centre line of a square perimeter should be the
  diagonal or the length. It has values: Options: 1: Centreline can be the diagonal; 2:
  Centreline will not be the diagonal
  Range=1,2
  Values=1,2
  Default=1
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('expmmw')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute expmmw
print("Running expmmw...")
dm_cmd.expmmw(
    perimin_i='t_input_file',  # required
    perimout_o='t_expmmw_out',  # required
    mmw_p='required_val',  # required
    # pinc_p=0.1,  # optional
    # mode_p=1,  # optional
    # nodiag_p=1,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("expmmw execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_expmmw_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")